In [ ]:
from PIL import Image
import subprocess
import os
import tempfile

def compress_image(input_path, filesize_threshold=500):
    """Function to compress an image using pillow and optionally ffmpeg, 
    to a maximum of the specified filesize threshold [KB]. It will compress 
    in multiple stages if necessary to meet the file size threshold."""

    # open an image with pillow
    with Image.open(input_path) as my_image:

        # original image height and width
        original_image_height = my_image.height
        original_image_width = my_image.width
        original_filesize = len(my_image.fp.read())/1024

        print(f"Original image size: height={original_image_height}, width={original_image_width}")
        print("The original filesize of image is: ", round(original_filesize), "KB")

        # resize the image
        new_image_width = 800
        my_image = my_image.resize((new_image_width,int(my_image.height * (new_image_width / original_image_width))))

        new_image_height = my_image.height
        
    with tempfile.TemporaryDirectory(dir="D:") as temp_dir:

        # save the image to the temporary directory based on input file-extension
        if input_path.endswith('.jpg'):
            suffix='.jpg'
        elif input_path.endswith('.jpeg'):
            suffix='.jpeg'
        elif input_path.endswith('.png'):
            suffix='.png'
        else:
            raise ValueError("Unknown format")
        
        resized_image_path = os.path.join(temp_dir, "resized_image"+suffix)
        my_image.save(resized_image_path)

        # Check new file size after resizing
        new_filesize = os.path.getsize(resized_image_path) / 1024
        print(f"Resized image size: height={new_image_height}, width={new_image_width}")
        print("The filesize of resized image is: ", round(new_filesize), "KB")

        # Done if threshold is met after resizing only, otherwise continue with ffmpeg compression steps
        if not new_filesize > filesize_threshold:
            os.replace(resized_image_path, input_path)

        else:
            print("Compressing further with ffmpeg...")
            if suffix == ".jpg":
                quality_range = range(1, 31, 1)
                max_compression_value = 31
                encoder = "-q:v"
                encoder_setting1 = ""
                encoder_setting2 = ""
            elif suffix == '.png':
                quality_range = range(0, 12, 1)
                max_compression_value = 100
                encoder = "-q:v"
                encoder_setting1 = "png"
                encoder_setting2 = "-compression_level"
            else:
                raise ValueError("Unknown format")
            
            for quality_value in quality_range:
                    compressed_image_path = os.path.join(temp_dir, "compressed_image"+suffix) 

                    ffmpeg_command = [
                        "ffmpeg", 
                        "-i",
                        resized_image_path,
                        "-y", 
                        encoder,
                        encoder_setting1,
                        encoder_setting2,
                        str(quality_value),
                    ]
                    ffmpeg_command.append(compressed_image_path)
                    print(ffmpeg_command)
                    # run ffmpeg using subprocess
                    print(f"Trying quality value: {quality_value} ...")
                    result = subprocess.run(ffmpeg_command, capture_output=True, text=True)
                    if result.returncode != 0:
                        print(f"FFmpeg failed with return code: {result.returncode}")
                        print("STDOUT:", result.stdout)
                        print("STDERR:", result.stderr)

                    # Check new file size after compression
                    new_filesize = os.path.getsize(compressed_image_path) / 1024
                    print(f"Quality_value: {quality_value}, filesize: {round(new_filesize)} KB")
                    
                    if 0 < new_filesize <= filesize_threshold:
                        print(f"Target reached with quality_value {quality_value}")
                        # Replace resized image file with compressed version
                        os.replace(compressed_image_path, input_path)
                        break
                    elif new_filesize > filesize_threshold and quality_value == max_compression_value:
                        raise ValueError("Output filesize still too big after maximum compression!")
                    elif new_filesize == 0:
                        raise ValueError("Output filesize is zero KB!")
                    else:
                        raise ValueError(f"Filesize: {round(new_filesize)} KB with quality value: {quality_value}")
    print("Done")

                
if __name__ == "__main__":

    input_folder_path = r"D:\Code\Project_Van_Hool\Media\dak"
    # Process all image files in the input folder
    for filename in os.listdir(input_folder_path):
        # Skip non-image files and files that are small enough already
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(input_folder_path, filename)
            filesize_threshold=500 # [KB]
            too_big = (os.path.getsize(file_path) / 1024) > filesize_threshold
            if too_big:
                print(f"Compressing: {filename}")
                try:
                    # Compress and overwrite original
                    compress_image(file_path, filesize_threshold)
                    print(f"✓ {filename} compressed")
                except Exception as e:
                    print(f"✗ Error with {filename}: {e}")
    
    print(f"All images in {input_folder_path} processed!")

In [23]:
subprocess.run(["ffmpeg", "-i", r'D:\Code\Project_Van_Hool\Media\dak\dak29.4.png', '-preset', 'medium', r'D:\Code\Project_Van_Hool\Media\dak\dak29.4.jpg', '-y'])

CompletedProcess(args=['ffmpeg', '-i', 'D:\\Code\\Project_Van_Hool\\Media\\dak\\dak29.4.png', '-preset', 'medium', 'D:\\Code\\Project_Van_Hool\\Media\\dak\\dak29.4.jpg', '-y'], returncode=0)